# Evaluation
Run every approach from the other notebooks over two datasets and compare:

* **duration**: mean wall-clock time per transcript (all calls for that approach)
* **cost**: estimated from token usage and the prices below
* **invalid JSON**: outputs that didn't parse or didn't match the schema (unknown application / action / parameter)
* **accuracy**: application, action, and exact match (application + action + every parameter)

Datasets live in `eval_data/`: `basic.json` covers every action and option, `hard.json` has pivots
("navigate to X... actually find a supermarket"), misleading words and indirect requests.

In [1]:
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display
from pydantic import BaseModel, Field
from typesafe_sdk import Choice, TypeSafeClient

from demo_helpers import VALID_APPLICATIONS

load_dotenv()
gemini = genai.Client()

GEMINI_MODEL = "gemini-3.5-flash-lite"
TYPESAFE_MODEL = "jev-latest"
MAX_WORKERS = 2  # transcripts evaluated in parallel (per approach)
MAX_RETRIES = 5  # retries on rate limits (429); the timer restarts on each attempt

## Pricing
USD per million tokens. **Check these against current price lists before quoting numbers.**
Gemini "thinking" tokens are billed as output. TypeSafe output tokens are currently free.

In [2]:
PRICING = {
    "gemini": {"input": 0.30, "output": 2.50},
    "typesafe": {"input": 0.042, "output": 0.00},
}

## Datasets

In [3]:
DATASETS = {
    "basic": json.load(open("eval_data/basic.json", encoding="utf-8")),
    "hard": json.load(open("eval_data/hard.json", encoding="utf-8")),
}
pd.DataFrame(
    [{"dataset": name, "id": ex["id"], "transcript": ex["transcript"],
      "expected": json.dumps(ex["expected"])} for name, exs in DATASETS.items() for ex in exs]
)

,dataset,id,transcript,expected
0,basic,maps-navigate-1,"Hey Google, navigate to Ara.","{""application"": ""maps"", ""action"": ""navigate"", ..."
1,basic,maps-navigate-walking,Give me walking directions to Hagley Park.,"{""application"": ""maps"", ""action"": ""navigate"", ..."
2,basic,maps-navigate-cycling,Take me to Sumner Beach by bike.,"{""application"": ""maps"", ""action"": ""navigate"", ..."
3,basic,maps-navigate-transit,How do I get to Christchurch Airport on public...,"{""application"": ""maps"", ""action"": ""navigate"", ..."
4,basic,maps-navigate-driving,Drive me to Riccarton Mall.,"{""application"": ""maps"", ""action"": ""navigate"", ..."
5,basic,maps-search,Find coffee shops near me.,"{""application"": ""maps"", ""action"": ""search_near..."
6,basic,maps-search-radius,Search for petrol stations within 5 kilometres.,"{""application"": ""maps"", ""action"": ""search_near..."
7,basic,maps-traffic,How's the traffic looking right now?,"{""application"": ""maps"", ""action"": ""show_traffi..."
8,basic,music-play-song,Play Bohemian Rhapsody.,"{""application"": ""music"", ""action"": ""play"", ""pa..."
9,basic,music-play-artist,Put on some Six60.,"{""application"": ""music"", ""action"": ""play"", ""pa..."


## Harness
Every approach is a function `transcript, usage -> dict`. It records token usage into `usage`
and returns `{"application", "action", "parameters"}`, raising `InvalidOutput` when the model's
output doesn't parse or doesn't match the schema.

In [4]:
class InvalidOutput(Exception):
    pass


@dataclass
class Usage:
    calls: list[tuple[str, int, int]] = field(default_factory=list)  # (provider, in, out)

    def gemini(self, response):
        m = response.usage_metadata
        out = (m.candidates_token_count or 0) + (m.thoughts_token_count or 0)
        self.calls.append(("gemini", m.prompt_token_count or 0, out))

    def typesafe(self, response):
        u = response.usage
        self.calls.append(("typesafe", u.input_tokens or 0, u.output_tokens or 0))

    @property
    def cost(self) -> float:
        return sum(
            (i * PRICING[p]["input"] + o * PRICING[p]["output"]) / 1e6 for p, i, o in self.calls
        )

    @property
    def tokens(self) -> int:
        return sum(i + o for _, i, o in self.calls)


def extract_json(text: str):
    """Pull a JSON object out of free text (markdown fences, preamble, ...)."""
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        pass
    match = re.search(r"\{.*\}", text or "", re.DOTALL)
    if not match:
        raise InvalidOutput("no JSON object in output")
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError as e:
        raise InvalidOutput(f"bad JSON: {e}")


def validate_flat(obj) -> dict:
    """Check a {target_application, target_action, additional_parameters} dict
    against VALID_APPLICATIONS and convert it to the canonical form."""
    if not isinstance(obj, dict):
        raise InvalidOutput("output is not an object")
    app, action = obj.get("target_application"), obj.get("target_action")
    params = obj.get("additional_parameters") or {}
    if app not in VALID_APPLICATIONS:
        raise InvalidOutput(f"unknown application {app!r}")
    if action not in VALID_APPLICATIONS[app]["actions"]:
        raise InvalidOutput(f"unknown action {app}.{action}")
    if not isinstance(params, dict):
        raise InvalidOutput("additional_parameters is not an object")
    unknown = set(params) - set(VALID_APPLICATIONS[app]["actions"][action])
    if unknown:
        raise InvalidOutput(f"unknown parameters {sorted(unknown)}")
    return {"application": app, "action": action,
            "parameters": {k: v for k, v in params.items() if v not in (None, "")}}

## Approaches
Shared prompts and schemas, copied from the individual notebooks.

In [5]:
APPS_JSON = json.dumps(VALID_APPLICATIONS, indent=2)
SCHEMA_STRING = """
{
    "target_application": "[application]",
    "target_action": "[action]",
    "additional_parameters": {
        "[parameter_name_1]": "[parameter_value_1]",
        ...
    }
}
"""
TOOL_NAME = "submit_parsed_command"


def naive_prompt(t):
    return f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n transcript: {t}"""


def just_ask_prompt(t):
    return f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
Do not include any additional information or preamble. \n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{APPS_JSON}\n
Return the response as json following the schema:\n{SCHEMA_STRING}

transcript: {t}"""


def schema_prompt(t, extra=""):
    return f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{APPS_JSON}\n{extra}

transcript: {t}"""


class FlatCommand(BaseModel):
    target_application: str = Field(description="The application the user wants to use")
    target_action: str = Field(description="The action to perform within that application")
    additional_parameters: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Parameters for the action mentioned in the transcript, as a mapping "
            "of parameter name (as listed for the chosen action) to value"
        ),
    )

In [6]:
def run_naive(transcript, usage):
    r = gemini.models.generate_content(model=GEMINI_MODEL, contents=naive_prompt(transcript))
    usage.gemini(r)
    return validate_flat(extract_json(r.text))


def run_just_ask(transcript, usage):
    r = gemini.models.generate_content(model=GEMINI_MODEL, contents=just_ask_prompt(transcript))
    usage.gemini(r)
    return validate_flat(extract_json(r.text))


def run_json_mode(transcript, usage):
    r = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=just_ask_prompt(transcript),
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    usage.gemini(r)
    return validate_flat(extract_json(r.text))


def run_json_schema_mode(transcript, usage):
    r = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=schema_prompt(transcript),
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_json_schema=FlatCommand.model_json_schema(),
        ),
    )
    usage.gemini(r)
    return validate_flat(r.parsed if r.parsed is not None else extract_json(r.text))


TOOL_CONFIG = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[types.FunctionDeclaration(
        name=TOOL_NAME,
        description="Submit the parsed command extracted from the transcript.",
        parameters_json_schema=FlatCommand.model_json_schema(),
    )])],
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(
        mode="ANY", allowed_function_names=[TOOL_NAME])),
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def run_tool_calling(transcript, usage):
    r = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=schema_prompt(transcript, f"\nSubmit your answer by calling the {TOOL_NAME} tool."),
        config=TOOL_CONFIG,
    )
    usage.gemini(r)
    if not r.function_calls:
        raise InvalidOutput("no tool call")
    return validate_flat(r.function_calls[0].args)

### JSON schema mode 2 (typed schema per application/action)

In [7]:
class Navigate(BaseModel):
    action: Literal["navigate"]
    destination: str
    mode: Literal["driving", "walking", "cycling", "transit"] | None = None

class SearchNearby(BaseModel):
    action: Literal["search_nearby"]
    query: str
    radius_km: float | None = None

class ShowTraffic(BaseModel):
    action: Literal["show_traffic"]

class Maps(BaseModel):
    """Navigation and location search"""
    application: Literal["maps"]
    command: Navigate | SearchNearby | ShowTraffic

class Play(BaseModel):
    action: Literal["play"]
    song: str | None = None
    artist: str | None = None
    playlist: str | None = None

class Pause(BaseModel):
    action: Literal["pause"]

class Skip(BaseModel):
    action: Literal["skip"]

class SetVolume(BaseModel):
    action: Literal["set_volume"]
    level: int = Field(ge=0, le=100)

class Music(BaseModel):
    """Music streaming and playback"""
    application: Literal["music"]
    command: Play | Pause | Skip | SetVolume

class SendMessage(BaseModel):
    action: Literal["send_message"]
    recipient: str
    body: str

class ReadLatest(BaseModel):
    action: Literal["read_latest"]
    sender: str | None = None

class Messages(BaseModel):
    """Send and read text messages"""
    application: Literal["messages"]
    command: SendMessage | ReadLatest

class CreateEvent(BaseModel):
    action: Literal["create_event"]
    title: str
    date: str = Field(description="YYYY-MM-DD")
    time: str | None = Field(default=None, description="HH:MM")
    duration_minutes: int | None = None

class ListEvents(BaseModel):
    action: Literal["list_events"]
    date: str = Field(description="YYYY-MM-DD")

class SetReminder(BaseModel):
    action: Literal["set_reminder"]
    text: str
    time: str = Field(description="HH:MM")

class Calendar(BaseModel):
    """Manage events and reminders"""
    application: Literal["calendar"]
    command: CreateEvent | ListEvents | SetReminder

class CurrentConditions(BaseModel):
    action: Literal["current_conditions"]
    location: str | None = None

class Forecast(BaseModel):
    action: Literal["forecast"]
    location: str | None = None
    days: int | None = None

class Weather(BaseModel):
    """Weather forecasts and conditions"""
    application: Literal["weather"]
    command: CurrentConditions | Forecast

class TypedCommand(BaseModel):
    target: Maps | Music | Messages | Calendar | Weather


def const_to_enum(schema):
    if isinstance(schema, dict):
        if "const" in schema:
            schema["enum"] = [schema.pop("const")]
        for value in schema.values():
            const_to_enum(value)
    elif isinstance(schema, list):
        for item in schema:
            const_to_enum(item)
    return schema


TYPED_SCHEMA = const_to_enum(TypedCommand.model_json_schema())


def run_json_schema_mode_2(transcript, usage):
    r = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=f"""For the following transcript, determine the target application the user
wants to use, the action to take, and any parameters for that action.

transcript: {transcript}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_json_schema=TYPED_SCHEMA),
    )
    usage.gemini(r)
    try:
        result = TypedCommand.model_validate(r.parsed if r.parsed is not None else extract_json(r.text))
    except Exception as e:
        raise InvalidOutput(str(e).splitlines()[0])
    command = result.target.command.model_dump(exclude_none=True)
    action = command.pop("action")
    return {"application": result.target.application, "action": action, "parameters": command}

### Decision models (TypeSafe)
These choose from fixed options, so their output is always valid. Models 1 and 2 can't
extract parameters, so they're scored on application/action only.

In [8]:
NONE_OPTION = "none"


def ts_choice(transcript_or_state, questions, usage):
    with TypeSafeClient(model=TYPESAFE_MODEL) as client:
        r = client.system_one(state=transcript_or_state, questions=questions)
    usage.typesafe(r)
    return {k: a.choice for k, a in r.answers.items()}


def pick_app(transcript, usage):
    return ts_choice(transcript, {"application": Choice(
        instructions="Which application does the user want to use?",
        criteria={n: a["description"] for n, a in VALID_APPLICATIONS.items()},
    )}, usage)["application"]


def pick_action(transcript, app_name, usage):
    app = VALID_APPLICATIONS[app_name]
    return ts_choice(transcript, {"action": Choice(
        instructions=(f"The user is using the {app_name} application "
                      f"({app['description']}). Which action do they want?"),
        criteria={a: a.replace("_", " ") for a in app["actions"]},
    )}, usage)["action"]


def run_decision_1(transcript, usage):
    criteria = {
        f"{app_name}.{action}": f"{app['description']}: {action.replace('_', ' ')}"
        for app_name, app in VALID_APPLICATIONS.items() for action in app["actions"]
    }
    choice = ts_choice(transcript, {"command": Choice(
        instructions="Which application and action does the user want?", criteria=criteria,
    )}, usage)["command"]
    app, action = choice.split(".", 1)
    return {"application": app, "action": action, "parameters": None}


def run_decision_2(transcript, usage):
    app = pick_app(transcript, usage)
    return {"application": app, "action": pick_action(transcript, app, usage), "parameters": None}


def run_decision_3(transcript, usage):
    app = pick_app(transcript, usage)
    action = pick_action(transcript, app, usage)
    parameters = VALID_APPLICATIONS[app]["actions"][action]
    extracted = {}
    if parameters:
        words = transcript.split()
        numbered = "\n".join(f"{i}: {w}" for i, w in enumerate(words))
        state = f"transcript: {transcript}\n\nwords (id: word):\n{numbered}"
        word_criteria = {str(i): w for i, w in enumerate(words)}
        word_criteria[NONE_OPTION] = "The parameter is not mentioned in the transcript"
        questions = {}
        for param, ptype in parameters.items():
            described = f'the "{param}" parameter ({ptype}) of the {app} {action} action'
            for end, which in (("phrase_start", "FIRST"), ("phrase_end", "LAST")):
                questions[f"{param}.{end}"] = Choice(
                    instructions=(f"Extract {described} as a phrase from the transcript. "
                                  f"Pick the ID of the {which} word of the phrase, "
                                  f'or "{NONE_OPTION}" if it is not mentioned.'),
                    criteria=word_criteria,
                )
        answers = ts_choice(state, questions, usage)
        for param in parameters:
            s, e = answers[f"{param}.phrase_start"], answers[f"{param}.phrase_end"]
            if NONE_OPTION in (s, e):
                continue
            first, last = sorted((int(s), int(e)))
            extracted[param] = " ".join(words[first:last + 1]).strip(".,!?")
    return {"application": app, "action": action, "parameters": extracted}

In [9]:
APPROACHES = {
    "naive": run_naive,
    "just ask": run_just_ask,
    "json mode": run_json_mode,
    "json schema mode": run_json_schema_mode,
    "tool calling": run_tool_calling,
    "json schema mode 2": run_json_schema_mode_2,
    "decision model 1": run_decision_1,
    "decision model 2": run_decision_2,
    "decision model 3": run_decision_3,
}

## Scoring
Parameter values are compared loosely: case, surrounding punctuation, a leading "the" and
whitespace are ignored, and numbers are compared numerically (`"40"` == `40`).

In [10]:
def norm(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        s = re.sub(r"\s+", " ", str(value).lower()).strip(" .,!?'\"")
        s = s.replace("\u2019", "'")
        return re.sub(r"^the ", "", s)


def params_match(predicted: dict, expected: dict) -> bool:
    if set(predicted) != set(expected):
        return False
    return all(norm(predicted[k]) == norm(expected[k]) for k in expected)


def is_rate_limit(e: Exception) -> bool:
    return "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)


def evaluate_one(approach, example) -> dict:
    for attempt in range(MAX_RETRIES + 1):
        row = attempt_one(approach, example)
        if not (row["error"] and "429" in row["error"]) or attempt == MAX_RETRIES:
            return row
        time.sleep(2 ** attempt * 5)


def attempt_one(approach, example) -> dict:
    usage = Usage()
    row = {"id": example["id"], "valid": True, "error": None, "output": None}
    start = time.perf_counter()
    try:
        out = approach(example["transcript"], usage)
        row["output"] = out
    except InvalidOutput as e:
        row["valid"], row["error"], out = False, f"invalid: {e}", None
    except Exception as e:  # API errors etc.: not the model's fault, but still a miss
        row["error"], out = ("429 " if is_rate_limit(e) else "") + f"{type(e).__name__}: {e}", None
    row["seconds"] = time.perf_counter() - start
    row["cost"], row["tokens"] = usage.cost, usage.tokens

    exp = example["expected"]
    row["app_ok"] = bool(out) and out["application"] == exp["application"]
    row["action_ok"] = row["app_ok"] and out["action"] == exp["action"]
    if out and out["parameters"] is not None:
        row["exact_ok"] = row["action_ok"] and params_match(out["parameters"], exp["parameters"])
    else:
        row["exact_ok"] = None if out else False  # None = approach can't extract parameters
    return row


def evaluate(approach, examples):
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        return list(pool.map(lambda ex: evaluate_one(approach, ex), examples))

## Run
Run everything (takes a few minutes), or pick a subset of approaches/datasets.

In [12]:
selected_approaches = list(APPROACHES)  # e.g. ["json schema mode", "decision model 3"]
selected_datasets = list(DATASETS)

rows = []
for ds in selected_datasets:
    for name in selected_approaches:
        start = time.perf_counter()
        results = evaluate(APPROACHES[name], DATASETS[ds])
        print(f"{ds:6} {name:20} {time.perf_counter() - start:6.1f}s")
        rows += [{"dataset": ds, "approach": name, **r} for r in results]

raw = pd.DataFrame(rows)

basic  naive                  14.8s
basic  just ask               17.4s
basic  json mode              18.3s
basic  json schema mode       15.4s
basic  tool calling           14.4s
basic  json schema mode 2     15.3s
basic  decision model 1        6.2s
basic  decision model 2       23.1s
basic  decision model 3       24.7s
hard   naive                   9.1s
hard   just ask                7.7s
hard   json mode               8.5s
hard   json schema mode        8.7s
hard   tool calling            8.3s
hard   json schema mode 2      9.5s
hard   decision model 1        3.7s
hard   decision model 2        7.8s
hard   decision model 3       10.7s


In [17]:
raw.to_csv("temp.csv")

## Results

In [13]:
def summarise(df):
    g = df.groupby(["dataset", "approach"], sort=False)
    out = pd.DataFrame({
        "n": g.size(),
        "mean time (s)": g["seconds"].mean(),
        "p95 time (s)": g["seconds"].quantile(0.95),
        "cost / 1k ($)": g["cost"].mean() * 1000,
        "tokens / call": g["tokens"].mean(),
        "invalid JSON": g["valid"].apply(lambda s: (~s).sum()),
        "API errors": g.apply(lambda d: (d["valid"] & d["error"].notna()).sum(), include_groups=False),
        "app acc": g["app_ok"].mean(),
        "action acc": g["action_ok"].mean(),
        "exact acc": g["exact_ok"].apply(lambda s: s.mean() if s.notna().any() else float("nan")),
    })
    return out


summary = summarise(raw)


def style(df):
    return (
        df.style
        .format({"mean time (s)": "{:.2f}", "p95 time (s)": "{:.2f}", "cost / 1k ($)": "${:.4f}",
                 "tokens / call": "{:,.0f}", "app acc": "{:.0%}", "action acc": "{:.0%}",
                 "exact acc": "{:.0%}"}, na_rep="n/a")
        .background_gradient(subset=["app acc", "action acc", "exact acc"], cmap="RdYlGn", vmin=0, vmax=1)
        .background_gradient(subset=["mean time (s)", "cost / 1k ($)"], cmap="Blues")
        .bar(subset=["invalid JSON"], color="#e8a0a0", vmin=0)
        .set_caption("Cost is per 1,000 transcripts. Exact = application + action + all parameters.")
    )


for ds in summary.index.get_level_values("dataset").unique():
    display(style(summary.loc[ds]).set_caption(f"<b>{ds}</b> dataset"))

,n,mean time (s),p95 time (s),cost / 1k ($),tokens / call,invalid JSON,API errors,app acc,action acc,exact acc
approach,,,,,,,,,,
naive,25,1.14,1.38,$0.1669,101,25,0,0%,0%,0%
just ask,25,1.35,1.30,$0.3070,673,0,0,100%,100%,100%
json mode,25,1.45,2.08,$0.3070,673,0,0,100%,100%,100%
json schema mode,25,1.19,1.38,$0.2812,595,0,0,100%,100%,100%
tool calling,25,1.14,1.37,$0.3295,777,0,0,100%,100%,100%
json schema mode 2,25,1.21,1.44,$0.1496,97,0,0,100%,100%,100%
decision model 1,25,0.49,0.59,$0.0241,721,0,0,100%,100%,n/a
decision model 2,25,1.83,3.33,$0.0301,812,0,0,100%,100%,n/a
decision model 3,25,1.95,3.08,$0.0670,"1,950",0,0,100%,100%,76%


,n,mean time (s),p95 time (s),cost / 1k ($),tokens / call,invalid JSON,API errors,app acc,action acc,exact acc
approach,,,,,,,,,,
naive,14,1.29,1.70,$0.2074,126,14,0,0%,0%,0%
just ask,14,1.10,1.23,$0.3150,685,0,0,100%,100%,100%
json mode,14,1.18,1.39,$0.3150,685,0,0,100%,100%,100%
json schema mode,14,1.20,1.36,$0.2897,608,0,0,100%,100%,100%
tool calling,14,1.14,1.28,$0.3364,789,0,0,100%,100%,100%
json schema mode 2,14,1.34,1.55,$0.1553,109,0,0,100%,100%,100%
decision model 1,14,0.53,0.62,$0.0245,731,0,0,100%,100%,n/a
decision model 2,14,1.10,1.20,$0.0310,833,0,0,93%,93%,n/a
decision model 3,14,1.51,2.03,$0.0824,"2,508",0,0,93%,93%,57%


### Basic vs hard: exact-match accuracy

In [14]:
pivot = summary["exact acc"].unstack("dataset")
pivot.style.format("{:.0%}", na_rep="n/a").background_gradient(cmap="RdYlGn", vmin=0, vmax=1, axis=None)

dataset,basic,hard
approach,,
naive,0%,0%
just ask,100%,100%
json mode,100%,100%
json schema mode,100%,100%
tool calling,100%,100%
json schema mode 2,100%,100%
decision model 1,n/a,n/a
decision model 2,n/a,n/a
decision model 3,76%,57%


## Failures
Every transcript an approach got wrong, with what it produced.

In [15]:
fails = raw[~raw["exact_ok"].fillna(raw["action_ok"]).astype(bool)].copy()
expected = {ex["id"]: ex for exs in DATASETS.values() for ex in exs}
fails["transcript"] = fails["id"].map(lambda i: expected[i]["transcript"])
fails["expected"] = fails["id"].map(lambda i: json.dumps(expected[i]["expected"]))
fails["got"] = fails.apply(lambda r: r["error"] if pd.notna(r["error"]) else json.dumps(r["output"]), axis=1)
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 500):
    display(fails[["dataset", "approach", "id", "transcript", "expected", "got"]])

,dataset,approach,id,transcript,expected,got
0,basic,naive,maps-navigate-1,"Hey Google, navigate to Ara.","{""application"": ""maps"", ""action"": ""navigate"", ""parameters"": {""destination"": ""Ara""}}",invalid: no JSON object in output
1,basic,naive,maps-navigate-walking,Give me walking directions to Hagley Park.,"{""application"": ""maps"", ""action"": ""navigate"", ""parameters"": {""destination"": ""Hagley Park"", ""mode"": ""walking""}}",invalid: no JSON object in output
2,basic,naive,maps-navigate-cycling,Take me to Sumner Beach by bike.,"{""application"": ""maps"", ""action"": ""navigate"", ""parameters"": {""destination"": ""Sumner Beach"", ""mode"": ""cycling""}}",invalid: no JSON object in output
3,basic,naive,maps-navigate-transit,How do I get to Christchurch Airport on public transport?,"{""application"": ""maps"", ""action"": ""navigate"", ""parameters"": {""destination"": ""Christchurch Airport"", ""mode"": ""transit""}}",invalid: no JSON object in output
4,basic,naive,maps-navigate-driving,Drive me to Riccarton Mall.,"{""application"": ""maps"", ""action"": ""navigate"", ""parameters"": {""destination"": ""Riccarton Mall"", ""mode"": ""driving""}}",invalid: no JSON object in output
5,basic,naive,maps-search,Find coffee shops near me.,"{""application"": ""maps"", ""action"": ""search_nearby"", ""parameters"": {""query"": ""coffee shops""}}",invalid: no JSON object in output
6,basic,naive,maps-search-radius,Search for petrol stations within 5 kilometres.,"{""application"": ""maps"", ""action"": ""search_nearby"", ""parameters"": {""query"": ""petrol stations"", ""radius_km"": 5}}",invalid: no JSON object in output
7,basic,naive,maps-traffic,How's the traffic looking right now?,"{""application"": ""maps"", ""action"": ""show_traffic"", ""parameters"": {}}",invalid: no JSON object in output
8,basic,naive,music-play-song,Play Bohemian Rhapsody.,"{""application"": ""music"", ""action"": ""play"", ""parameters"": {""song"": ""Bohemian Rhapsody""}}",invalid: no JSON object in output
9,basic,naive,music-play-artist,Put on some Six60.,"{""application"": ""music"", ""action"": ""play"", ""parameters"": {""artist"": ""Six60""}}",invalid: no JSON object in output
